# Bayesian Optimisation: ERK Oscillation Frequency

Uses the `BOptGPAX` inter-phase agent with **batch acquisition** to find
the optimal optogenetic stimulation parameters that maximise the fraction
of oscillating cells (ERK-KTR cytoplasmic/nuclear ratio oscillations).

**Steerable parameters (BO optimises):**
- `stim_exposure` -- base stimulation light exposure in ms (50--500, step 25)
- `ramp` -- per-frame exposure increment in ms (0--50, step 5).

**Covariates (observed, not controlled):**
- `n_cells` -- number of tracked cells in the FOV
- `optortk_expression` -- mean optocheck (mCitrine) intensity per FOV

**Objective:** maximise `frac_oscillating` -- fraction of cells classified as
oscillating per FOV, using a pre-trained sliding-window XGBoost classifier.

**Phased acquisition with FOV finding (NEW):** each phase visits a fresh
batch of **6 wells** chosen from a user-supplied list of 60 wells. A
`FOVFinderAgent` picks **3 FOVs per well** (= 18 FOVs / phase) using the
plate calibration from the MDA plate widget — random candidates are
scanned, filtered by cell count, and farthest-point sampled to avoid
overlap. The FOV finder is a **pre-phase agent** plugged into a generic
`ComposedAgent` that drives the BO inter-phase agent for **10 phases**
total (60 wells / 6 wells per phase = 10 phases). The GP accumulates
observations across all 10 phases.

**Batch BO:** Each phase tests **3 conditions** simultaneously across the
18 fresh FOVs (6 FOVs per condition), using sequential greedy acquisition
with local penalisation for within-batch diversity.

**GP model:** ExactGP with Matern kernel (MCMC inference).


In [ ]:
import os
import time
import logging
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Suppress JAX/XLA debug messages
os.environ["JAX_LOG_COMPILES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax

jax.config.update("jax_log_compiles", False)
for _name in list(logging.Logger.manager.loggerDict):
    if "jax" in _name or "absl" in _name:
        logging.getLogger(_name).setLevel(logging.WARNING)

# ---------------------------------------------------------------------------
# gpax/numpyro compatibility shim
# ---------------------------------------------------------------------------
# numpyro >= 0.20 removed haiku support from numpyro.contrib.module, but
# gpax 0.1.9 eagerly imports viDKL via gpax/models/__init__.py, which tries
# `from numpyro.contrib.module import random_haiku_module, haiku_module`.
# We don't use viDKL (only ExactGP, viGP, viSparseGP), so we install stubs
# that raise only if viDKL is actually called. This MUST run before any
# `import gpax` (including the lazy ones inside the BO agent classes).
import numpyro.contrib.module as _ncm


def _haiku_unavailable(*_args, **_kwargs):
    raise NotImplementedError(
        "haiku support was removed from numpyro >= 0.20. "
        "viDKL is not available. Use ExactGP, viGP, or viSparseGP."
    )


if not hasattr(_ncm, "random_haiku_module"):
    _ncm.random_haiku_module = _haiku_unavailable
if not hasattr(_ncm, "haiku_module"):
    _ncm.haiku_module = _haiku_unavailable
# ---------------------------------------------------------------------------

from faro.core.data_structures import (
    Channel,
    PowerChannel,
    RTMSequence,
    SegmentationMethod,
)
from faro.core.controller import Controller
from faro.core.pipeline import ImageProcessingPipeline
from faro.agents.bo_optimization import (
    BOptGPAX,
    BO_Parameter,
    BO_Objective,
    BO_Covariate,
)
import faro.core.utils as utils

## Microscope & pipeline setup

**Microscope:** Jungfrau (no DMD -- full-FOV stimulation).

**Channels:**
- Imaging: miRFP (nuclear marker) + mScarlet3 (ERK-KTR reporter)
- Stimulation: CyanStim (optogenetic activation, power 10)
- Optocheck: mCitrine (verify optoRTK expression, last frame only)

**Pipeline:** CellposeV4 segmentation -> ERK-KTR FE -> Trackpy tracking
-> OptoCheckFE on reference frames. Full-FOV stimulation (no DMD).

In [ ]:
from faro.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

In [ ]:
# --- Experiment parameters ---
TIME_BETWEEN_TIMESTEPS = 60  # seconds (1 frame/min)
N_FRAMES_BASELINE = 10  # 10 min baseline
FIRST_FRAME_STIM = N_FRAMES_BASELINE
LAST_FRAME_STIM = 60  # frames 10-59 = 50 min stimulation
N_FRAMES = 61  # 10 baseline + 50 stim + 1 optocheck

# --- Phased FOV layout ---
N_WELLS_PER_PHASE = 6  # 6 wells per phase
FOVS_PER_WELL = 3  # 3 fresh FOVs per well per phase
N_FOVS = N_WELLS_PER_PHASE * FOVS_PER_WELL  # 18 FOVs per phase
N_CONDITIONS_PER_ITER = 3  # batch BO: 3 conditions tested per phase
FOVS_PER_CONDITION = N_FOVS // N_CONDITIONS_PER_ITER  # 6
N_PHASES = 10  # 60 wells supplied -> 10 phases

# --- Plate calibration ---
# Path to a WellPlatePlan JSON saved by the pymmcore-widgets MDA plate widget.
PLATE_CALIBRATION_PATH = r"E:\\Alex\\plate_calibration_96well.json"  # <-- UPDATE

# --- Wells to use (consumed in chunks of N_WELLS_PER_PHASE per phase) ---
# Supply N_PHASES * N_WELLS_PER_PHASE = 60 wells, in the order you want them
# visited.  The FOV finder pops the next chunk on every phase.
WELLS = [
    # row B (12)
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B9",
    "B10",
    "B11",
    # row C (12)
    "C2",
    "C3",
    "C4",
    "C5",
    "C6",
    "C7",
    "C8",
    "C9",
    "C10",
    "C11",
    # row D (12)
    "D2",
    "D3",
    "D4",
    "D5",
    "D6",
    "D7",
    "D8",
    "D9",
    "D10",
    "D11",
    # row E (12)
    "E2",
    "E3",
    "E4",
    "E5",
    "E6",
    "E7",
    "E8",
    "E9",
    "E10",
    "E11",
    # row F (12)
    "F2",
    "F3",
    "F4",
    "F5",
    "F6",
    "F7",
    "F8",
    "F9",
    "F10",
    "F11",
    # row G (10)
    "G2",
    "G3",
    "G4",
    "G5",
    "G6",
    "G7",
    "G8",
    "G9",
    "G10",
    "G11",
]
assert (
    len(WELLS) >= N_PHASES * N_WELLS_PER_PHASE
), f"Need at least {N_PHASES * N_WELLS_PER_PHASE} wells, got {len(WELLS)}"

# --- FOV finder knobs ---
FOV_BORDER_UM = 400.0  # keep candidate positions away from well edge
FOV_MIN_CELLS = 20  # min cell count for a candidate to be valid
FOV_N_CANDIDATES_PER_WELL = 8  # oversample to give farthest-point room

# --- Storage ---
base_path = "E:\\Alex"
experiment_name = "bo_erk_oscillation"
path = os.path.join(base_path, experiment_name)

# --- Stimulation channel (base -- exposure is overridden by BO) ---
stim_channel = PowerChannel(
    config="CyanStim",
    exposure=100,
    group="TTL_ERK",
    power=10,
)

# --- Imaging channels (acquired every timepoint AND used by FOV finder) ---
imaging_channels = (
    PowerChannel(config="miRFP", exposure=125, group="TTL_ERK", power=95),
    PowerChannel(config="mScarlet3", exposure=125, group="TTL_ERK", power=95),
)

# --- Optocheck channel (acquired on last frame only) ---
optocheck_channel = PowerChannel(
    config="mCitrine",
    exposure=600,
    group="TTL_ERK",
    power=99,
)

In [ ]:
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.segmentation.cellpose_v4 import CellposeV4

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=[
        SegmentationMethod(
            name="labels",
            segmentation_class=CellposeV4(),
            use_channel=0,  # segment on miRFP (nuclear marker)
            save_tracked=True,
        )
    ],
    feature_extractor=FE_ErkKtr("labels"),
    tracker=TrackerTrackpy(search_range=50),
    stimulator=StimWholeFOV(),
    feature_extractor_ref=OptoCheckFE(used_mask="labels"),
)

from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(storage_path=path)

## FOV selection (per-phase, automated)

Instead of manually selecting 18 positions in napari, a `FOVFinderAgent`
(plugged into the generic `ComposedAgent`) picks fresh positions before
**every phase**:

1. The agent loads the plate calibration (`WellPlatePlan` JSON saved by
   the pymmcore-widgets MDA plate widget).
2. It pops the next `N_WELLS_PER_PHASE` (= 6) wells from `WELLS`.
3. For each well it generates `FOV_N_CANDIDATES_PER_WELL` (= 8) random
   candidate positions, kept `FOV_BORDER_UM` µm away from the well edge.
4. It snaps the candidates with the user's imaging channels via
   `mic.run_mda` and segments the result with `CellposeV4`.
5. Candidates with `< FOV_MIN_CELLS` cells are dropped; the remaining
   ones are reduced to `FOVS_PER_WELL` (= 3) per well by greedy
   farthest-point sampling so the picked FOVs do not overlap.

The result (18 fresh FOVs / phase) is fed straight into
`OscillationBO.run_one_phase`, which is wired up by the `ComposedAgent`
in the next sections.

You only need to wire up the segmentator here; the `FOVFinderAgent` is
constructed in the *Configure and run* section together with the BO
agent and the composed driver.

In [ ]:
# Segmentator used by the FOV finder for cell counting on the seg channel.
# We segment the first imaging channel (miRFP, nuclear marker).
from faro.segmentation.cellpose_v4 import CellposeV4

fov_finder_segmentator = CellposeV4()

## Oscillation classifier

Load the pre-trained sliding-window oscillation classifier. This model uses
FFT, ACF, and time-domain features extracted from the ERK-KTR `cnr_median`
trace to classify each window as oscillating or not.

A cell is classified as **oscillating** if:
- max oscillation probability across all windows >= 0.85
- longest run of consecutive predicted_osc==1 windows >= 4

In [ ]:
import joblib

# --- Load pre-trained oscillation classifier ---
CLASSIFIER_PATH = r"path\to\oscillation_model.joblib"  # <-- UPDATE THIS PATH

model_data = joblib.load(CLASSIFIER_PATH)
osc_clf = model_data["clf"]
osc_scaler = model_data["scaler"]
osc_feature_cols = model_data["feature_cols"]
osc_cfg = model_data["config"]
osc_cfg["window_size"] = model_data["window_size"]
osc_cfg["window_step"] = model_data["window_step"]

# Import the predict_trace function from the classifier script
# os.getcwd() is typically the repo root in VSCode
_classifier_script = os.path.join(
    os.getcwd(), "temp2", "apply_oscillation_classifier.py"
)
_spec = importlib.util.spec_from_file_location("osc_classifier", _classifier_script)
_osc_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_osc_module)
predict_trace = _osc_module.predict_trace

# --- Oscillation thresholds ---
MIN_OSC_PROBABILITY = 0.85
MIN_CONSECUTIVE_WINDOWS = 4

print(f"Loaded oscillation classifier from {CLASSIFIER_PATH}")
print(f"  Window: {osc_cfg['window_size']} steps, stride {osc_cfg['window_step']}")
print(
    f"  Thresholds: prob >= {MIN_OSC_PROBABILITY}, consecutive >= {MIN_CONSECUTIVE_WINDOWS}"
)

## Define the Batch BO agent

Subclass `BOptGPAX` and override:
- `_create_events_for_batch` -- builds RTMEvents for 18 FOVs with 3 conditions
  (6 FOVs each), applying per-frame ramped stim exposure
- `_preprocess_results` -- runs the oscillation classifier on each cell's
  `cnr_median` trace and computes the fraction of oscillating cells per FOV
- `_select_batch_parameters` -- selects 3 conditions per iteration via
  sequential greedy acquisition with local penalisation
- `run` -- overrides the base loop for batch BO (3 conditions per iteration)

In [ ]:
from IPython.display import display, clear_output


class OscillationBO(BOptGPAX):
    """Batch BO agent for ERK oscillation optimisation.

    Each BO phase tests 3 conditions simultaneously across 18 FOVs
    (6 FOVs per condition).  When driven by :class:`ComposedAgent` the
    18 FOVs are *fresh* every phase (chosen by ``FOVFinderAgent``); the
    GP accumulates observations across all phases.

    The class implements both:

    * ``run()`` (legacy) -- runs all ``n_iterations`` phases on the
      *same* FOVs (the original behaviour).
    * ``run_one_phase(phase_id, fov_positions, fovs)`` -- composable hook
      called by :class:`ComposedAgent`.  Each call updates the FOVs for
      this phase, runs one batch BO iteration, and accumulates results
      in ``self.df_results``.  FOV indices are offset by
      ``phase_id * n_fovs_per_phase`` so each phase gets fresh FOV ids
      (no controller "FOV position changed" warnings, fresh per-FOV
      tracker state).
    """

    def __init__(
        self,
        *,
        fov_positions,
        n_frames,
        first_frame_stim,
        last_frame_stim,
        time_between_timesteps,
        imaging_channels,
        stim_channel,
        optocheck_channel,
        n_conditions_per_iter,
        osc_clf,
        osc_scaler,
        osc_feature_cols,
        osc_cfg,
        osc_predict_fn,
        min_osc_probability=0.85,
        min_consecutive_windows=4,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.fov_positions = fov_positions
        self.n_frames = n_frames
        self.first_frame_stim = first_frame_stim
        self.last_frame_stim = last_frame_stim
        self.time_between_timesteps = time_between_timesteps
        self.imaging_channels = imaging_channels
        self.stim_channel = stim_channel
        self.optocheck_channel = optocheck_channel
        self.n_conditions_per_iter = n_conditions_per_iter
        self.fovs_per_condition = len(fov_positions) // n_conditions_per_iter

        self.osc_clf = osc_clf
        self.osc_scaler = osc_scaler
        self.osc_feature_cols = osc_feature_cols
        self.osc_cfg = osc_cfg
        self.osc_predict_fn = osc_predict_fn
        self.min_osc_probability = min_osc_probability
        self.min_consecutive_windows = min_consecutive_windows

        self._phase_counter = 0
        self._current_condition_map = {}
        # When driven by ComposedAgent, FOV indices are shifted by this much
        # so each phase has globally unique FOV ids.
        self._fov_index_offset = 0

    # ------------------------------------------------------------------
    # Event creation
    # ------------------------------------------------------------------

    def _create_events_for_cycle(self, parameters: dict) -> list:
        raise NotImplementedError("Use _create_events_for_batch for batch BO")

    def _create_events_for_batch(self, param_list: list[dict]) -> list:
        """Build RTMEvents for all conditions in one batch iteration.

        Creates 3 RTMSequences (one per condition, each with 6 FOVs),
        remaps FOV indices to be globally unique, merges, and applies
        FOV batching.  When driven by ``run_one_phase`` the FOV indices
        are *also* shifted by ``self._fov_index_offset`` so each phase
        gets fresh FOV ids.
        """
        phase_id = self._phase_counter
        all_events = []

        stim_frames = range(self.first_frame_stim, self.last_frame_stim)
        n_stim_frames = len(stim_frames)

        self._current_condition_map = {}

        for cond_idx, params in enumerate(param_list):
            start = cond_idx * self.fovs_per_condition
            end = start + self.fovs_per_condition
            cond_positions = self.fov_positions[start:end]

            # _current_condition_map is keyed by GLOBAL fov index so that
            # _preprocess_results (which iterates self.fovs == globals)
            # can look up the right BO params for each FOV.
            for fov_idx in range(start, end):
                self._current_condition_map[fov_idx + self._fov_index_offset] = params

            base_exp = float(params["stim_exposure"])
            ramp = float(params["ramp"])
            exposures = tuple(base_exp + ramp * i for i in range(n_stim_frames))

            acq = RTMSequence(
                time_plan={
                    "interval": self.time_between_timesteps,
                    "loops": self.n_frames,
                },
                stage_positions=cond_positions,
                channels=self.imaging_channels,
                stim_channels=(self.stim_channel,),
                stim_frames=stim_frames,
                stim_exposure=exposures,
                ref_channels=(self.optocheck_channel,),
                ref_frames=frozenset({self.n_frames - 1}),
                rtm_metadata={
                    "phase_name": f"BO_iter_{phase_id}_cond_{cond_idx}",
                    "phase_id": phase_id,
                    "condition_idx": cond_idx,
                    "stim_exposure": base_exp,
                    "ramp": ramp,
                },
            )

            # Remap local p indices (0..5) to GLOBAL FOV indices.
            #   condition offset:   cond_idx * fovs_per_condition  -> 0, 6, 12
            #   phase    offset:   self._fov_index_offset          -> 0, 18, 36, ...
            p_offset = cond_idx * self.fovs_per_condition + self._fov_index_offset
            for ev in acq:
                local_p = ev.index.get("p", 0)
                all_events.append(
                    ev.model_copy(
                        update={"index": {**dict(ev.index), "p": local_p + p_offset}}
                    )
                )

        all_events = utils.apply_fov_batching(all_events, time_per_fov=2.0)

        print(f"  Created {len(all_events)} events for phase {phase_id}")
        for i, p in enumerate(param_list):
            exp_start = p["stim_exposure"]
            exp_end = exp_start + p["ramp"] * (n_stim_frames - 1)
            fov_start = i * self.fovs_per_condition + self._fov_index_offset
            fov_end = fov_start + self.fovs_per_condition - 1
            print(
                f"    Cond {i} (FOVs {fov_start}-{fov_end}): "
                f"stim_exposure={exp_start:.0f}ms, ramp={p['ramp']:.0f}ms/frame "
                f"(final exposure: {exp_end:.0f}ms)"
            )
        return all_events

    # ------------------------------------------------------------------
    # Oscillation classification
    # ------------------------------------------------------------------

    def _is_cell_oscillating(self, windows: list[dict]) -> bool:
        if not windows:
            return False
        probs = [w["osc_probability"] for w in windows]
        preds = [w["predicted_osc"] for w in windows]
        if max(probs) < self.min_osc_probability:
            return False
        max_consec = 0
        current = 0
        for p in preds:
            if p == 1:
                current += 1
                max_consec = max(max_consec, current)
            else:
                current = 0
        return max_consec >= self.min_consecutive_windows

    # ------------------------------------------------------------------
    # Result preprocessing
    # ------------------------------------------------------------------

    def _preprocess_results(self, fov_tracks: dict, phase_id: int) -> pd.DataFrame:
        """Classify oscillations and compute per-FOV metrics."""
        results = []
        for fov_idx, df_tracks in fov_tracks.items():
            if df_tracks.empty or "particle" not in df_tracks.columns:
                continue

            # --- Filter to current phase only ---
            if "phase_id" in df_tracks.columns:
                df_phase = df_tracks[df_tracks["phase_id"] == phase_id]
                if df_phase.empty:
                    continue
            else:
                df_phase = df_tracks

            params = self._current_condition_map.get(fov_idx)
            if params is None:
                print(f"  Warning: no condition mapping for FOV {fov_idx}, skipping")
                continue

            n_cells = df_phase["particle"].nunique()
            n_oscillating = 0

            optortk_vals = []
            if "ref_mean_intensity" in df_phase.columns:
                optortk_vals = (
                    df_phase.groupby("particle")["ref_mean_intensity"]
                    .first()
                    .dropna()
                    .values
                )

            for _, grp in df_phase.groupby("particle"):
                grp = grp.sort_values("timestep")
                if "cnr_median" not in grp.columns or len(grp) < 20:
                    continue
                x = grp["timestep"].values.astype(float)
                y = grp["cnr_median"].values.astype(float)
                windows = self.osc_predict_fn(
                    x,
                    y,
                    self.osc_clf,
                    self.osc_scaler,
                    self.osc_feature_cols,
                    self.osc_cfg,
                )
                if self._is_cell_oscillating(windows):
                    n_oscillating += 1

            if n_cells < 5:
                print(f"  Warning: FOV {fov_idx} has only {n_cells} cells, skipping")
                continue

            frac_oscillating = n_oscillating / n_cells
            optortk_expression = (
                float(np.median(optortk_vals)) if len(optortk_vals) > 0 else 0.0
            )

            results.append(
                {
                    "stim_exposure": params["stim_exposure"],
                    "ramp": params["ramp"],
                    "n_cells": float(n_cells),
                    "optortk_expression": optortk_expression,
                    "frac_oscillating": frac_oscillating,
                }
            )

        if not results:
            print(f"Warning: no valid FOVs in phase {phase_id}")
            return pd.DataFrame()

        df = pd.DataFrame(results)
        print(
            f"  Phase {phase_id}: {len(df)} FOVs, "
            f"mean frac_oscillating={df['frac_oscillating'].mean():.4f}, "
            f"max={df['frac_oscillating'].max():.4f}"
        )
        return df

    # ------------------------------------------------------------------
    # Live visualization (called after each iteration)
    # ------------------------------------------------------------------

    def _plot_live(self, df_results: pd.DataFrame, iteration_label: str):
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(iteration_label, fontsize=13, fontweight="bold")

        sc = axes[0].scatter(
            df_results["stim_exposure"],
            df_results["ramp"],
            c=df_results["frac_oscillating"],
            cmap="viridis",
            s=40,
            edgecolors="k",
            linewidths=0.5,
        )
        axes[0].set_xlabel("stim_exposure (ms)")
        axes[0].set_ylabel("ramp (ms/frame)")
        axes[0].set_title("Measured frac_oscillating per FOV")
        fig.colorbar(sc, ax=axes[0], label="frac_oscillating")

        if self._iteration_means:
            means = np.array(self._iteration_means)
            cumulative_best = np.maximum.accumulate(means)
            axes[1].plot(
                np.arange(len(means)), means, "o-", alpha=0.5, label="iteration mean"
            )
            axes[1].plot(
                np.arange(len(means)),
                cumulative_best,
                "s-",
                color="red",
                label="cumulative best",
            )
            axes[1].set_xlabel("Iteration")
            axes[1].set_ylabel("frac_oscillating")
            axes[1].set_title("BO convergence")
            axes[1].legend()

        cond_agg = (
            df_results.groupby(["stim_exposure", "ramp"])
            .agg(
                mean_frac=("frac_oscillating", "mean"),
                n_fovs=("frac_oscillating", "count"),
            )
            .reset_index()
        )
        sc2 = axes[2].scatter(
            cond_agg["stim_exposure"],
            cond_agg["ramp"],
            c=cond_agg["mean_frac"],
            s=cond_agg["n_fovs"] * 30,
            cmap="viridis",
            edgecolors="k",
            linewidths=0.5,
        )
        axes[2].set_xlabel("stim_exposure (ms)")
        axes[2].set_ylabel("ramp (ms/frame)")
        axes[2].set_title("Mean per condition (size=n_fovs)")
        fig.colorbar(sc2, ax=axes[2], label="mean frac_oscillating")
        plt.tight_layout()
        plt.show()

        if not cond_agg.empty:
            best = cond_agg.loc[cond_agg["mean_frac"].idxmax()]
            print(
                f"  Current best: stim_exposure={best['stim_exposure']:.0f}ms, "
                f"ramp={best['ramp']:.0f}ms/frame -> "
                f"mean frac_oscillating={best['mean_frac']:.4f}"
            )

    def _save_checkpoint(self, df_results: pd.DataFrame):
        ckpt_path = os.path.join(self.storage_path, "bo_results_checkpoint.parquet")
        df_results.to_parquet(ckpt_path, index=False)

    # ------------------------------------------------------------------
    # Batch acquisition: select 3 conditions per iteration
    # ------------------------------------------------------------------

    def _select_batch_parameters(self, df_results, n_conditions=3):
        if df_results.empty or len(df_results) < 4:
            initial, _ = self._select_initial_samples(k=n_conditions)
            if isinstance(initial, dict):
                initial = [initial]
            return initial[:n_conditions]

        selected = []
        orig_penalty = self.penalty
        orig_penalty_factor = self.penalty_factor

        for i in range(n_conditions):
            if i > 0:
                self.penalty = "inverse_distance"
                self.penalty_factor = 2.0
            try:
                params = self._determine_next_parameters(
                    df_results, verbose=(i == 0 and self.verbose)
                )
                selected.append(params)
            except Exception as e:
                print(f"  Warning: batch selection {i} failed: {e}")
                if len(self.x_unmeasured) > 0:
                    idx = self._rng.integers(len(self.x_unmeasured))
                    params = {
                        p.name: float(self.x_unmeasured[idx, j])
                        for j, p in enumerate(self.parameters_to_optimize)
                    }
                    selected.append(params)

        self.penalty = orig_penalty
        self.penalty_factor = orig_penalty_factor
        return selected

    # ------------------------------------------------------------------
    # Composable per-phase entry point (used by ComposedAgent)
    # ------------------------------------------------------------------

    def run_one_phase(
        self,
        phase_id: int,
        fov_positions: list | None = None,
        fovs: list[int] | None = None,
    ) -> dict | None:
        """Run a single batch BO phase on a fresh set of FOV positions.

        Called by :class:`ComposedAgent` after each pre-phase agent
        (e.g. ``FOVFinderAgent``) produces fresh FOVs.

        Args:
            phase_id: Zero-based phase index.  Used as the experiment's
                ``phase_id`` metadata, the FOV-index offset, and to
                decide between ``run_experiment`` (==0) and
                ``continue_experiment`` (>0).
            fov_positions: List of stage positions for this phase.  Must
                have the same length as the original ``fov_positions``
                that was passed to ``__init__`` (so the
                ``fovs_per_condition`` slicing still works).
            fovs: Ignored — overridden internally so each phase gets
                globally-unique FOV indices
                ``[phase_id * n, phase_id * n + 1, ...]``.
        """
        self._ensure_results_df()

        if fov_positions is None:
            raise ValueError(
                "OscillationBO.run_one_phase requires fov_positions "
                "(supplied per-phase by the FOVFinderAgent)."
            )
        if len(fov_positions) != len(self.fov_positions):
            raise ValueError(
                f"fov_positions length mismatch: got {len(fov_positions)}, "
                f"expected {len(self.fov_positions)} (set at __init__)."
            )

        n_per_phase = len(fov_positions)
        self.fov_positions = list(fov_positions)
        self._fov_index_offset = phase_id * n_per_phase
        self.fovs = list(
            range(self._fov_index_offset, self._fov_index_offset + n_per_phase)
        )
        self._phase_counter = phase_id

        # --- Select batch params (initial spread or BO acquisition) ---
        if self.df_results.empty or len(self.df_results) < 4:
            initial_params, _ = self._select_initial_samples(
                k=self.n_conditions_per_iter
            )
            if isinstance(initial_params, dict):
                initial_params = [initial_params]
            batch_params = initial_params[: self.n_conditions_per_iter]
            print(f"=== Phase {phase_id}: initial batch ===")
        else:
            batch_params = self._select_batch_parameters(
                self.df_results, n_conditions=self.n_conditions_per_iter
            )
            print(f"=== Phase {phase_id}: BO-selected batch ===")

        for i, p in enumerate(batch_params):
            print(f"  Condition {i}: {p}")

        # --- Build events and run ---
        events = self._create_events_for_batch(batch_params)
        if phase_id == 0:
            self.controller.run_experiment(events, validate=False)
        else:
            self.controller.continue_experiment(events, validate=False)

        # --- Wait + collect results ---
        self._wait_for_pipeline(timeout=3600)
        tracks = {fov: self.read_tracks(fov, phase_id=phase_id) for fov in self.fovs}
        df_new = self._preprocess_results(tracks, phase_id=phase_id)

        if not df_new.empty:
            self.df_results = pd.concat([self.df_results, df_new], ignore_index=True)
            self._iteration_means.append(
                float(df_new[self.objective_metric.name].mean())
            )
            self._save_checkpoint(self.df_results)
            self._plot_live(self.df_results, f"After phase {phase_id + 1}/{N_PHASES}")

        if not self.df_results.empty:
            self.x = self._extract_x_from_df(self.df_results)
            self.y = self._extract_y_from_df(self.df_results)

        self.iteration += 1
        return {"params": batch_params, "df_new": df_new, "phase_id": phase_id}

    # ------------------------------------------------------------------
    # Legacy entry point: run all phases on the same FOVs
    # ------------------------------------------------------------------

    def run(self) -> None:
        if not self.fovs:
            raise ValueError("No FOVs configured. Call add_fovs() before run().")

        df_results = pd.DataFrame()

        # --- Initial iteration: spread points (no GP yet) ---
        initial_params, _ = self._select_initial_samples(k=self.n_conditions_per_iter)
        if isinstance(initial_params, dict):
            initial_params = [initial_params]
        initial_params = initial_params[: self.n_conditions_per_iter]

        print(f"=== Initial batch: {self.n_conditions_per_iter} conditions ===")
        for i, p in enumerate(initial_params):
            print(f"  Condition {i}: {p}")

        phase_id = self._phase_counter
        events = self._create_events_for_batch(initial_params)
        self.controller.run_experiment(events, validate=False)

        self._wait_for_pipeline(timeout=3600)
        tracks = {fov: self.read_tracks(fov, phase_id=phase_id) for fov in self.fovs}
        df_new = self._preprocess_results(tracks, phase_id=phase_id)
        self._phase_counter += 1
        if not df_new.empty:
            df_results = pd.concat([df_results, df_new], ignore_index=True)
            self._iteration_means.append(
                float(df_new[self.objective_metric.name].mean())
            )
            self._save_checkpoint(df_results)
            self._plot_live(df_results, "After initial batch")

        # --- BO iterations ---
        for e in range(self.n_iterations):
            print(f"\n{'='*60}")
            print(f"=== BO Iteration {e + 1}/{self.n_iterations} ===")
            print(f"  Data so far: {len(df_results)} observations")
            print(f"{'='*60}")

            batch_params = self._select_batch_parameters(
                df_results, n_conditions=self.n_conditions_per_iter
            )
            for i, p in enumerate(batch_params):
                print(f"  Selected condition {i}: {p}")

            phase_id = self._phase_counter
            events = self._create_events_for_batch(batch_params)
            self.controller.continue_experiment(events, validate=False)

            self._wait_for_pipeline(timeout=3600)
            tracks = {
                fov: self.read_tracks(fov, phase_id=phase_id) for fov in self.fovs
            }
            df_new = self._preprocess_results(tracks, phase_id=phase_id)
            self._phase_counter += 1
            if not df_new.empty:
                df_results = pd.concat([df_results, df_new], ignore_index=True)
                self._iteration_means.append(
                    float(df_new[self.objective_metric.name].mean())
                )
                self._save_checkpoint(df_results)
                self._plot_live(
                    df_results,
                    f"After BO iteration {e + 1}/{self.n_iterations}",
                )
            self.iteration += 1

        if not df_results.empty:
            self.x = self._extract_x_from_df(df_results)
            self.y = self._extract_y_from_df(df_results)
        self.df_results = df_results

        self.controller.finish_experiment()
        print(
            f"\nBO complete. {len(df_results)} total observations across "
            f"{self._phase_counter} phases."
        )

## Per-cell variant: `OscillationBOSparse` (viSparseGP)

The `OscillationBO` class above aggregates to one observation per FOV
(`frac_oscillating`). With 18 FOVs / iteration, ExactGP handles this fine.

For **per-cell BO** -- where each cell becomes one observation -- the
dataset grows to thousands of points per iteration (e.g. 18 FOVs * 100
cells = 1800/iter). At ~20,000 observations after 11 iterations,
ExactGP's O(n^3) MCMC becomes infeasible.

`OscillationBOSparse` solves this with:
- **Variational inference** (`viSparseGP` or `viGP`) instead of MCMC
- **Analytic EI** (closed-form, using mean/variance) instead of MC sampling
- **Per-cell observations**: each cell -> one row in df_results
- **Cell-level covariate**: `cell_optortk_expression` (per-cell intensity,
  not FOV median)
- **Continuous objective**: `max_osc_probability` (max prob across
  classifier windows, in [0, 1])

When to use which:
- `OscillationBO`: small datasets (<500 obs), per-FOV `frac_oscillating`,
  ExactGP+MCMC -- gives proper posterior uncertainty
- `OscillationBOSparse`: large datasets (1000-100k obs), per-cell
  `max_osc_probability`, viSparseGP -- scales to single-cell resolution

In [ ]:
class OscillationBOSparse(OscillationBO):
    """Per-cell BO using variational sparse GP.

    Subclasses :class:`OscillationBO` and overrides:

    * :meth:`_preprocess_results` -- one row per cell instead of per FOV
    * :meth:`_determine_next_parameters` -- viSparseGP + analytic EI/UCB

    All event creation, batch selection, run loop, and live visualisation
    are inherited unchanged from :class:`OscillationBO`.

    Args:
        gp_backend: ``"vi_sparse"`` (default, scales best) or ``"vi"``
            (full variational, no inducing points -- use for medium datasets).
        inducing_points_ratio: Fraction of training points used as inducing
            points (only for ``vi_sparse``).  0.05 = 5% inducing points.
        num_svi_steps: SVI optimisation steps for the variational fit.
        svi_step_size: Adam step size for SVI.
        gp_batch_size: Batch size for GP prediction (memory control).
    """

    def __init__(
        self,
        *,
        gp_backend: str = "vi_sparse",
        inducing_points_ratio: float = 0.1,
        num_svi_steps: int = 1000,
        svi_step_size: float = 5e-3,
        gp_batch_size: int = 2000,
        **kwargs,
    ):
        super().__init__(**kwargs)
        if gp_backend not in {"vi_sparse", "vi"}:
            raise ValueError(
                f"Unknown gp_backend={gp_backend!r}, use 'vi_sparse' or 'vi'"
            )
        self.gp_backend = gp_backend
        self.inducing_points_ratio = inducing_points_ratio
        self.num_svi_steps = num_svi_steps
        self.svi_step_size = svi_step_size
        self.gp_batch_size = gp_batch_size

    # ------------------------------------------------------------------
    # Per-cell result preprocessing (overrides FOV-aggregated version)
    # ------------------------------------------------------------------

    def _preprocess_results(self, fov_tracks: dict, phase_id: int) -> pd.DataFrame:
        """Return one row per cell with cell-level features.

        Columns:
            stim_exposure, ramp           -- BO parameters (FOV-level)
            n_cells                       -- FOV-level covariate
            cell_optortk_expression       -- per-cell covariate
            max_osc_probability           -- per-cell objective
            fov, particle                 -- bookkeeping
        """
        rows = []
        for fov_idx, df_tracks in fov_tracks.items():
            if df_tracks.empty or "particle" not in df_tracks.columns:
                continue

            # Filter to current phase only
            if "phase_id" in df_tracks.columns:
                df_phase = df_tracks[df_tracks["phase_id"] == phase_id]
                if df_phase.empty:
                    continue
            else:
                df_phase = df_tracks

            params = self._current_condition_map.get(fov_idx)
            if params is None:
                print(f"  Warning: no condition mapping for FOV {fov_idx}, skipping")
                continue

            n_cells_fov = df_phase["particle"].nunique()
            if n_cells_fov < 5:
                print(
                    f"  Warning: FOV {fov_idx} has only {n_cells_fov} cells, skipping"
                )
                continue

            for particle_id, grp in df_phase.groupby("particle"):
                grp = grp.sort_values("timestep")
                if "cnr_median" not in grp.columns or len(grp) < 20:
                    continue

                x_t = grp["timestep"].values.astype(float)
                y_trace = grp["cnr_median"].values.astype(float)

                windows = self.osc_predict_fn(
                    x_t,
                    y_trace,
                    self.osc_clf,
                    self.osc_scaler,
                    self.osc_feature_cols,
                    self.osc_cfg,
                )
                if not windows:
                    continue

                max_osc_prob = float(max(w["osc_probability"] for w in windows))

                # Per-cell optoRTK expression (first non-NaN ref intensity)
                cell_optortk = 0.0
                if "ref_mean_intensity" in grp.columns:
                    vals = grp["ref_mean_intensity"].dropna()
                    if len(vals) > 0:
                        cell_optortk = float(vals.iloc[0])

                rows.append(
                    {
                        "stim_exposure": params["stim_exposure"],
                        "ramp": params["ramp"],
                        "n_cells": float(n_cells_fov),
                        "cell_optortk_expression": cell_optortk,
                        "max_osc_probability": max_osc_prob,
                        "fov": int(fov_idx),
                        "particle": int(particle_id),
                    }
                )

        if not rows:
            print(f"Warning: no valid cells in phase {phase_id}")
            return pd.DataFrame()

        df = pd.DataFrame(rows)
        print(
            f"  Phase {phase_id}: {len(df)} cells across {df['fov'].nunique()} FOVs, "
            f"mean max_osc_probability={df['max_osc_probability'].mean():.4f}, "
            f"max={df['max_osc_probability'].max():.4f}"
        )
        return df

    # ------------------------------------------------------------------
    # Override _plot_live to use the per-cell objective name
    # ------------------------------------------------------------------

    def _plot_live(self, df_results: pd.DataFrame, iteration_label: str):
        obj_col = self.objective_metric.name
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(iteration_label, fontsize=13, fontweight="bold")

        sc = axes[0].scatter(
            df_results["stim_exposure"],
            df_results["ramp"],
            c=df_results[obj_col],
            cmap="viridis",
            s=10,
            alpha=0.4,
            edgecolors="none",
        )
        axes[0].set_xlabel("stim_exposure (ms)")
        axes[0].set_ylabel("ramp (ms/frame)")
        axes[0].set_title(f"Per-cell {obj_col}")
        fig.colorbar(sc, ax=axes[0], label=obj_col)

        if self._iteration_means:
            means = np.array(self._iteration_means)
            cumulative_best = np.maximum.accumulate(means)
            axes[1].plot(
                np.arange(len(means)), means, "o-", alpha=0.5, label="iteration mean"
            )
            axes[1].plot(
                np.arange(len(means)),
                cumulative_best,
                "s-",
                color="red",
                label="cumulative best",
            )
            axes[1].set_xlabel("Iteration")
            axes[1].set_ylabel(obj_col)
            axes[1].set_title("BO convergence (mean per iteration)")
            axes[1].legend()

        cond_agg = (
            df_results.groupby(["stim_exposure", "ramp"])
            .agg(mean_obj=(obj_col, "mean"), n_cells=(obj_col, "count"))
            .reset_index()
        )
        sc2 = axes[2].scatter(
            cond_agg["stim_exposure"],
            cond_agg["ramp"],
            c=cond_agg["mean_obj"],
            s=np.sqrt(cond_agg["n_cells"]) * 8,
            cmap="viridis",
            edgecolors="k",
            linewidths=0.5,
        )
        axes[2].set_xlabel("stim_exposure (ms)")
        axes[2].set_ylabel("ramp (ms/frame)")
        axes[2].set_title(f"Mean per condition (size~sqrt(n_cells))")
        fig.colorbar(sc2, ax=axes[2], label=f"mean {obj_col}")
        plt.tight_layout()
        plt.show()

        if not cond_agg.empty:
            best = cond_agg.loc[cond_agg["mean_obj"].idxmax()]
            print(
                f"  Current best: stim_exposure={best['stim_exposure']:.0f}ms, "
                f"ramp={best['ramp']:.0f}ms/frame -> "
                f"mean {obj_col}={best['mean_obj']:.4f} "
                f"({int(best['n_cells'])} cells)"
            )

    # ------------------------------------------------------------------
    # viSparseGP fit + analytic EI/UCB (overrides ExactGP+MC)
    # ------------------------------------------------------------------

    def _determine_next_parameters(self, df_results, verbose=False):
        """Fit viSparseGP/viGP and compute analytic acquisition.

        viSparseGP returns (mean, variance) instead of posterior samples,
        so EI/UCB use closed-form expressions, not Monte Carlo.
        """
        import gpax
        import gpax.utils
        from scipy.stats import norm

        x_scaler = StandardScalerBounds()
        bounds, log_scale = self._get_bounds_and_log_scale()
        y_scaler = StandardScalerBounds()
        y_log_scale = [self.objective_metric.log_scale]

        x_raw = self._extract_x_from_df(df_results)
        y_raw = self._extract_y_from_df(df_results)

        x = x_scaler.fit_transform(x_raw, bounds=bounds, log_scale=log_scale)
        y = y_scaler.fit_transform(y_raw, bounds=None, log_scale=y_log_scale)
        x_np = np.asarray(x)
        y_flat = np.asarray(y).flatten()

        rng_key, rng_key_predict = gpax.utils.get_keys()

        # ----- Fit variational GP -----
        print(
            f"  Fitting {self.gp_backend} on {x_np.shape[0]} observations "
            f"({x_np.shape[1]}-D)..."
        )
        if self.gp_backend == "vi_sparse":
            gp_model = gpax.viSparseGP(input_dim=x_np.shape[1], kernel="Matern")
            gp_model.fit(
                rng_key,
                X=x_np,
                y=y_flat,
                inducing_points_ratio=self.inducing_points_ratio,
                num_steps=self.num_svi_steps,
                step_size=self.svi_step_size,
                progress_bar=verbose,
                print_summary=False,
            )
        else:  # "vi"
            gp_model = gpax.viGP(input_dim=x_np.shape[1], kernel="Matern")
            gp_model.fit(
                rng_key,
                X=x_np,
                y=y_flat,
                num_steps=self.num_svi_steps,
                step_size=self.svi_step_size,
                progress_bar=verbose,
                print_summary=False,
            )

        self.model = gp_model
        self._x_scaler = x_scaler
        self._y_scaler = y_scaler
        self._rng_key_predict = rng_key_predict

        x_grid_ctrl = self.x_unmeasured.copy()
        n_grid = x_grid_ctrl.shape[0]
        n_ctrl = len(self.parameters_to_optimize)
        current_xi = self._current_ei_xi()
        maximize = self.objective_metric.goal == "maximize"

        # ----- Build prediction grid -----
        if len(self.bo_covariates) > 0:
            cov_samples_list = []
            for covariate in self.bo_covariates:
                cov_vals = np.asarray(df_results[covariate.name].values, dtype=float)
                if cov_vals.size == 0:
                    cov_samples = np.array([0.0])
                else:
                    cov_samples = self._rng.choice(
                        cov_vals, size=self.n_cov_samples, replace=True
                    )
                cov_samples_list.append(cov_samples)

            if len(cov_samples_list) == 1:
                cov_grid = cov_samples_list[0].reshape(-1, 1)
            else:
                cov_mesh = np.meshgrid(*cov_samples_list, indexing="ij")
                cov_grid = np.stack([m.ravel() for m in cov_mesh], axis=1)

            n_mc = cov_grid.shape[0]
            x_repeated = np.repeat(x_grid_ctrl, n_mc, axis=0)
            c_tiled = np.tile(cov_grid, (n_grid, 1))
            x_full = np.hstack([x_repeated, c_tiled])
            x_full_scaled = np.asarray(x_scaler.transform(x_full))

            print(
                f"  Predicting {x_full_scaled.shape[0]} points "
                f"({n_grid} grid x {n_mc} cov samples)..."
            )
            mean_pred, var_pred = gp_model.predict_in_batches(
                rng_key_predict,
                x_full_scaled,
                batch_size=self.gp_batch_size,
                noiseless=True,
            )
            mean_matrix = np.asarray(mean_pred).reshape(n_grid, n_mc)
            var_matrix = np.asarray(var_pred).reshape(n_grid, n_mc)
            sigma_matrix = np.sqrt(np.maximum(var_matrix, 1e-12))

            if self.acquisition_function == "ei":
                # Best mean per covariate scenario (analog of base class behavior)
                if maximize:
                    best_f_per_cov = np.max(mean_matrix, axis=0)
                else:
                    best_f_per_cov = np.min(mean_matrix, axis=0)
                u = (
                    mean_matrix - (best_f_per_cov[None, :] + current_xi)
                ) / sigma_matrix
                if not maximize:
                    u = -u
                # Closed-form EI: sigma * (u * Phi(u) + phi(u))
                acq_matrix = sigma_matrix * (u * norm.cdf(u) + norm.pdf(u))
            else:  # UCB
                beta_sqrt = np.sqrt(self.ucb_beta)
                if maximize:
                    acq_matrix = mean_matrix + beta_sqrt * sigma_matrix
                else:
                    acq_matrix = -(mean_matrix - beta_sqrt * sigma_matrix)

            acq_matrix = np.where(np.isfinite(acq_matrix), acq_matrix, 0.0)
            # Marginalize over covariate samples
            acq_values_total = np.mean(acq_matrix, axis=1)
        else:
            x_total_scaled = np.asarray(x_scaler.transform(x_grid_ctrl))
            print(f"  Predicting {x_total_scaled.shape[0]} grid points...")
            mean_pred, var_pred = gp_model.predict_in_batches(
                rng_key_predict,
                x_total_scaled,
                batch_size=self.gp_batch_size,
                noiseless=True,
            )
            mean = np.asarray(mean_pred)
            sigma = np.sqrt(np.maximum(np.asarray(var_pred), 1e-12))

            if self.acquisition_function == "ei":
                best_f_scaled = float(np.max(y_flat) if maximize else np.min(y_flat))
                u = (mean - (best_f_scaled + current_xi)) / sigma
                if not maximize:
                    u = -u
                acq_values_total = sigma * (u * norm.cdf(u) + norm.pdf(u))
            else:
                beta_sqrt = np.sqrt(self.ucb_beta)
                if maximize:
                    acq_values_total = mean + beta_sqrt * sigma
                else:
                    acq_values_total = -(mean - beta_sqrt * sigma)
            acq_values_total = np.where(
                np.isfinite(acq_values_total), acq_values_total, 0.0
            )

        # ----- Apply penalty (for batch diversity) -----
        if self.penalty is not None and self.x_performed_experiments is not None:
            recent_ctrl_raw = self.x_performed_experiments[:, :n_ctrl]
            ctrl_log = (
                x_scaler.log_scale[:n_ctrl] if x_scaler.log_scale is not None else None
            )
            ctrl_mean = np.asarray(x_scaler.mean_)[:n_ctrl]
            ctrl_std = np.asarray(x_scaler.std_)[:n_ctrl]

            def _scale_ctrl(X):
                X = np.asarray(X, dtype=float).copy()
                if ctrl_log is not None:
                    for i, use_log in enumerate(ctrl_log):
                        if use_log:
                            X[:, i] = np.log(X[:, i])
                return (X - ctrl_mean) / ctrl_std

            x_grid_ctrl_scaled = _scale_ctrl(x_grid_ctrl)
            recent_ctrl_scaled = _scale_ctrl(recent_ctrl_raw)

            if self.penalty == "delta":
                for rp in recent_ctrl_scaled:
                    distances = np.linalg.norm(x_grid_ctrl_scaled - rp, axis=1)
                    mask = distances < 1e-8
                    acq_values_total = np.where(mask, -np.inf, acq_values_total)
            elif self.penalty == "inverse_distance":
                min_distances = np.full(x_grid_ctrl_scaled.shape[0], np.inf)
                for rp in recent_ctrl_scaled:
                    distances = np.linalg.norm(x_grid_ctrl_scaled - rp, axis=1)
                    min_distances = np.minimum(min_distances, distances)
                penalty_term = 1.0 / (min_distances + 1e-6)
                acq_values_total = acq_values_total / (
                    1.0 + self.penalty_factor * penalty_term
                )

        # ----- Select argmax and update state -----
        next_idx = int(np.argmax(acq_values_total))
        next_parameters = np.asarray(x_grid_ctrl[next_idx])

        self.x_unmeasured = np.delete(self.x_unmeasured, next_idx, axis=0)
        self.x_performed_experiments = (
            np.concatenate(
                [self.x_performed_experiments, next_parameters.reshape(1, -1)],
                axis=0,
            )
            if self.x_performed_experiments is not None
            else next_parameters.reshape(1, -1)
        )

        return {
            param.name: next_parameters[i]
            for i, param in enumerate(self.parameters_to_optimize)
        }

In [ ]:
# --- Configuration for the per-cell sparse-GP variant ---
# Use this instead of the per-FOV cell above when you want single-cell BO.
# Wired up the same way: a fresh FOVFinderAgent + a ComposedAgent so that
# each phase visits 18 fresh FOVs from a fresh batch of 6 wells.

bo_params_sparse = [
    BO_Parameter(name="stim_exposure", bounds=(50.0, 500.0), spacing=25.0),
    BO_Parameter(name="ramp", bounds=(0.0, 50.0), spacing=5.0),
]

# Cell-level optoRTK expression (per-cell ref_mean_intensity, not FOV median)
bo_covariates_sparse = [
    BO_Covariate(name="n_cells"),
    BO_Covariate(name="cell_optortk_expression"),
]

# Per-cell objective: max oscillation probability across windows
bo_objective_sparse = BO_Objective(name="max_osc_probability", goal="maximize")

fov_finder_sparse = FOVFinderAgent(
    microscope=mic,
    well_plate_plan=PLATE_CALIBRATION_PATH,
    wells=WELLS,
    wells_per_phase=N_WELLS_PER_PHASE,
    fovs_per_well=FOVS_PER_WELL,
    n_candidates_per_well=FOV_N_CANDIDATES_PER_WELL,
    border_um=FOV_BORDER_UM,
    min_cells=FOV_MIN_CELLS,
    imaging_channels=imaging_channels,
    segmentator=fov_finder_segmentator,
    seg_channel_index=0,
    random_seed=42,
)

agent_sparse = OscillationBOSparse(
    storage_path=path,
    fov_positions=[
        FovPosition(x=0.0, y=0.0, z=None, name=f"placeholder_{i}")
        for i in range(N_FOVS)
    ],
    n_frames=N_FRAMES,
    first_frame_stim=FIRST_FRAME_STIM,
    last_frame_stim=LAST_FRAME_STIM,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    imaging_channels=imaging_channels,
    stim_channel=stim_channel,
    optocheck_channel=optocheck_channel,
    n_conditions_per_iter=N_CONDITIONS_PER_ITER,
    osc_clf=osc_clf,
    osc_scaler=osc_scaler,
    osc_feature_cols=osc_feature_cols,
    osc_cfg=osc_cfg,
    osc_predict_fn=predict_trace,
    min_osc_probability=MIN_OSC_PROBABILITY,
    min_consecutive_windows=MIN_CONSECUTIVE_WINDOWS,
    parameters_to_optimize=bo_params_sparse,
    objective_metric=bo_objective_sparse,
    bo_covariates=bo_covariates_sparse,
    n_iterations=N_PHASES,
    acquisition_function="ei",
    n_cov_samples=20,
    ei_xi=0.2,
    ei_xi_final=0.01,
    ei_xi_decay_fraction=0.7,
    verbose=True,
    # Sparse GP-specific
    gp_backend="vi_sparse",
    inducing_points_ratio=0.05,
    num_svi_steps=1500,
    svi_step_size=5e-3,
    gp_batch_size=2000,
)

composed_agent_sparse = ComposedAgent(
    inner_agent=agent_sparse,
    pre_phase_agents=[fov_finder_sparse],
    n_phases=N_PHASES,
)

# Create a separate controller (don't reuse `ctrl` from the per-FOV path)
ctrl_sparse = Controller(mic, pipeline, writer=writer, agent=composed_agent_sparse)

print(f"Per-cell sparse BO configured")
print(f"  GP backend: {agent_sparse.gp_backend}")
print(f"  Inducing points ratio: {agent_sparse.inducing_points_ratio}")
print(f"  SVI steps: {agent_sparse.num_svi_steps}")
print(
    f"  Phases: {N_PHASES} x {N_FOVS} FOVs/phase = "
    f"{N_PHASES * N_FOVS} FOV observations max"
)
print(
    f"  Expected obs (per-cell): ~50-100 cells/FOV * "
    f"{N_PHASES * N_FOVS} FOVs = ~{N_PHASES * N_FOVS * 75} cells"
)

# To run the sparse-cell experiment instead of the per-FOV one:
#     composed_agent_sparse.run()

## Configure and run Bayesian Optimisation

**Parameter space:**
- `stim_exposure`: 50--500 ms (step 25) = 19 levels
- `ramp`: 0--50 ms/frame (step 5) = 11 levels
- Total grid: 19 x 11 = 209 candidate conditions

**Covariates:** `n_cells` and `optortk_expression` are observed per FOV and
marginalised over during acquisition function evaluation.

**Budget:** With 3 conditions/iteration and ~10 BO iterations (+ 1 initial),
we test ~33 unique conditions (6 FOVs each) = ~198 total FOV observations.

In [ ]:
from faro.core.utils import FovPosition
from faro.agents import FOVFinderAgent, ComposedAgent

# --- BO parameters ---
bo_params = [
    BO_Parameter(name="stim_exposure", bounds=(50.0, 500.0), spacing=25.0),
    BO_Parameter(name="ramp", bounds=(0.0, 50.0), spacing=5.0),
]

# --- Covariates (observed, not controlled) ---
bo_covariates = [
    BO_Covariate(name="n_cells"),
    BO_Covariate(name="optortk_expression"),
]

# --- Objective ---
bo_objective = BO_Objective(name="frac_oscillating", goal="maximize")

# --- Pre-phase agent: FOV finder ---
# It loads the plate calibration, scans candidates inside each well, segments
# them, and farthest-point selects FOVS_PER_WELL valid positions per well.
# A fresh batch of N_WELLS_PER_PHASE wells is consumed per phase.
fov_finder = FOVFinderAgent(
    microscope=mic,
    well_plate_plan=PLATE_CALIBRATION_PATH,
    wells=WELLS,
    wells_per_phase=N_WELLS_PER_PHASE,
    fovs_per_well=FOVS_PER_WELL,
    n_candidates_per_well=FOV_N_CANDIDATES_PER_WELL,
    border_um=FOV_BORDER_UM,
    min_cells=FOV_MIN_CELLS,
    imaging_channels=imaging_channels,
    segmentator=fov_finder_segmentator,
    seg_channel_index=0,  # segment on miRFP (nuclear marker)
    random_seed=42,
)
print(
    f"FOVFinder: {len(WELLS)} wells queued -> "
    f"{fov_finder.n_remaining_phases} phases @ {N_WELLS_PER_PHASE} wells/phase"
)

# --- BO agent ---
# fov_positions is a placeholder of the right LENGTH; the composed agent
# overwrites it with fresh positions on every phase via run_one_phase.
placeholder_positions = [
    FovPosition(x=0.0, y=0.0, z=None, name=f"placeholder_{i}") for i in range(N_FOVS)
]

agent = OscillationBO(
    storage_path=path,
    fov_positions=placeholder_positions,  # length must equal N_FOVS
    n_frames=N_FRAMES,
    first_frame_stim=FIRST_FRAME_STIM,
    last_frame_stim=LAST_FRAME_STIM,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    imaging_channels=imaging_channels,
    stim_channel=stim_channel,
    optocheck_channel=optocheck_channel,
    n_conditions_per_iter=N_CONDITIONS_PER_ITER,
    osc_clf=osc_clf,
    osc_scaler=osc_scaler,
    osc_feature_cols=osc_feature_cols,
    osc_cfg=osc_cfg,
    osc_predict_fn=predict_trace,
    min_osc_probability=MIN_OSC_PROBABILITY,
    min_consecutive_windows=MIN_CONSECUTIVE_WINDOWS,
    parameters_to_optimize=bo_params,
    objective_metric=bo_objective,
    bo_covariates=bo_covariates,
    n_iterations=N_PHASES,  # used for EI exploration decay schedule
    acquisition_function="ei",
    n_cov_samples=20,
    ei_xi=0.2,
    ei_xi_final=0.01,
    ei_xi_decay_fraction=0.7,
    verbose=True,
)

# --- Composed agent: drives FOV finder + BO for N_PHASES phases ---
composed_agent = ComposedAgent(
    inner_agent=agent,
    pre_phase_agents=[fov_finder],
    n_phases=N_PHASES,
)

# --- Create controller (wires controller into the composed agent and,
# transparently, into the inner BO agent) ---
ctrl = Controller(mic, pipeline, writer=writer, agent=composed_agent)

print(f"Parameter grid: {len(agent.x_total_linespace)} candidate conditions")
print(f"Phases: {N_PHASES}  (1 initial-spread batch + {N_PHASES - 1} BO batches)")
print(
    f"Conditions per phase: {N_CONDITIONS_PER_ITER}  "
    f"FOVs per condition: {FOVS_PER_CONDITION}  Total FOVs/phase: {N_FOVS}"
)
print(f"Total FOV observations after {N_PHASES} phases: " f"~{N_PHASES * N_FOVS}")

In [ ]:
# --- Run the composed (FOV finder + BO) loop ---
# Each phase: FOVFinder picks 18 fresh FOVs (3 FOVs in 6 fresh wells), then
# OscillationBO runs one batch BO iteration over those FOVs.  The GP
# accumulates observations across all phases.
composed_agent.run()

In [ ]:
# Post-processing: merge per-FOV tracks
utils.generate_exp_data_from_tracks(path)

## Visualise results

In [ ]:
if agent.df_results is not None and not agent.df_results.empty:
    df = agent.df_results

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Scatter: stim_exposure vs ramp, colored by frac_oscillating
    sc = axes[0].scatter(
        df["stim_exposure"],
        df["ramp"],
        c=df["frac_oscillating"],
        cmap="viridis",
        s=40,
        edgecolors="k",
        linewidths=0.5,
    )
    axes[0].set_xlabel("stim_exposure (ms)")
    axes[0].set_ylabel("ramp (ms/frame)")
    axes[0].set_title("Measured frac_oscillating per FOV")
    fig.colorbar(sc, ax=axes[0], label="frac_oscillating")

    # 2. Convergence: best frac_oscillating over iterations
    if agent._iteration_means:
        means = np.array(agent._iteration_means)
        cumulative_best = np.maximum.accumulate(means)
        axes[1].plot(means, "o-", alpha=0.5, label="iteration mean")
        axes[1].plot(cumulative_best, "s-", color="red", label="cumulative best")
        axes[1].set_xlabel("Iteration")
        axes[1].set_ylabel("frac_oscillating")
        axes[1].set_title("BO convergence")
        axes[1].legend()

    # 3. Per-condition aggregation: mean frac_oscillating per (stim_exposure, ramp)
    cond_agg = (
        df.groupby(["stim_exposure", "ramp"])
        .agg(
            mean_frac=("frac_oscillating", "mean"),
            n_fovs=("frac_oscillating", "count"),
        )
        .reset_index()
    )
    sc2 = axes[2].scatter(
        cond_agg["stim_exposure"],
        cond_agg["ramp"],
        c=cond_agg["mean_frac"],
        s=cond_agg["n_fovs"] * 30,
        cmap="viridis",
        edgecolors="k",
        linewidths=0.5,
    )
    axes[2].set_xlabel("stim_exposure (ms)")
    axes[2].set_ylabel("ramp (ms/frame)")
    axes[2].set_title("Mean frac_oscillating per condition\n(size = n_fovs)")
    fig.colorbar(sc2, ax=axes[2], label="mean frac_oscillating")

    plt.tight_layout()
    plt.show()

    # Print best condition
    best_idx = cond_agg["mean_frac"].idxmax()
    best = cond_agg.loc[best_idx]
    print(f"\nBest condition (by mean across FOVs):")
    print(f"  stim_exposure = {best['stim_exposure']:.0f} ms")
    print(f"  ramp = {best['ramp']:.0f} ms/frame")
    print(f"  mean frac_oscillating = {best['mean_frac']:.4f}")
    print(f"  tested on {int(best['n_fovs'])} FOVs")
else:
    print("No data collected yet.")

In [ ]:
# GP-predicted landscape (marginalised over covariates at their median values)
import gpax
import gpax.utils

if agent.model is not None and agent.x is not None:
    rng_key, rng_key_pred = gpax.utils.get_keys()

    # Create prediction grid over controllable parameters
    exp_vals = np.linspace(50.0, 500.0, 20)
    ramp_vals = np.linspace(0.0, 50.0, 15)

    # Fix covariates at their observed medians
    df = agent.df_results
    median_n_cells = df["n_cells"].median()
    median_optortk = df["optortk_expression"].median()

    grid = np.array(
        [[e, r, median_n_cells, median_optortk] for e in exp_vals for r in ramp_vals]
    )
    grid_scaled = agent._x_scaler.transform(grid)
    y_pred_scaled, _ = agent.model.predict(rng_key_pred, grid_scaled, noiseless=True)
    y_pred = agent._y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    # Reshape for contour plot
    Y_grid = np.asarray(y_pred).reshape(len(exp_vals), len(ramp_vals))

    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    cf = ax.contourf(ramp_vals, exp_vals, Y_grid, levels=20, cmap="viridis")
    fig.colorbar(cf, ax=ax, label="predicted frac_oscillating")

    # Overlay measured points
    ax.scatter(
        df["ramp"],
        df["stim_exposure"],
        c=df["frac_oscillating"],
        cmap="viridis",
        s=30,
        edgecolors="white",
        linewidths=0.5,
        zorder=5,
    )

    # Mark GP optimum
    opt_idx = int(np.argmax(np.asarray(y_pred)))
    ax.scatter(
        grid[opt_idx, 1],
        grid[opt_idx, 0],
        c="red",
        s=200,
        marker="*",
        edgecolors="black",
        linewidths=2,
        zorder=10,
        label=f"GP optimum: exp={grid[opt_idx, 0]:.0f}ms, ramp={grid[opt_idx, 1]:.0f}ms",
    )

    ax.set_xlabel("ramp (ms/frame)")
    ax.set_ylabel("stim_exposure (ms)")
    ax.set_title(
        f"GP-predicted frac_oscillating landscape\n"
        f"(covariates fixed at medians: n_cells={median_n_cells:.0f}, "
        f"optortk={median_optortk:.0f})"
    )
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    print(f"GP-predicted optimum:")
    print(f"  stim_exposure = {grid[opt_idx, 0]:.0f} ms")
    print(f"  ramp = {grid[opt_idx, 1]:.0f} ms/frame")
    print(f"  predicted frac_oscillating = {float(y_pred[opt_idx]):.4f}")

In [ ]:
# Covariate effects: how n_cells and optoRTK expression affect oscillation fraction
if agent.df_results is not None and not agent.df_results.empty:
    df = agent.df_results

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(df["n_cells"], df["frac_oscillating"], alpha=0.6, s=20)
    axes[0].set_xlabel("n_cells per FOV")
    axes[0].set_ylabel("frac_oscillating")
    axes[0].set_title("Cell density vs oscillation fraction")

    axes[1].scatter(df["optortk_expression"], df["frac_oscillating"], alpha=0.6, s=20)
    axes[1].set_xlabel("optoRTK expression (mCitrine intensity)")
    axes[1].set_ylabel("frac_oscillating")
    axes[1].set_title("optoRTK expression vs oscillation fraction")

    plt.tight_layout()
    plt.show()